<a href="https://colab.research.google.com/github/xorbotz/AI-LABS-STUFF/blob/main/Prompting_with_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- applied-genai-header -->
<div style="background: linear-gradient(135deg, #1e4b8f 0%, #2d6cb8 100%); color: white; padding: 24px; border-radius: 12px; font-family: 'Segoe UI', sans-serif; margin-bottom: 16px;">
  <div style="font-size: 12px; opacity: 0.85; letter-spacing: 1.5px; text-transform: uppercase;">Applied Generative AI · IE 5373</div>
  <h1 style="margin: 8px 0 4px 0; font-size: 28px; font-weight: 600;">Prompting with the OpenAI API</h1>
  <div style="font-size: 14px; opacity: 0.9;">First steps: chat completions, system/user/assistant roles, parameters</div>
  <div style="margin-top: 12px; font-size: 12px; opacity: 0.8;">Prof. Mohammad Dehghani · Northeastern University</div>
</div>

> **📌 Note on models.** This lab references specific LLM versions (e.g. `gpt-5`, `gpt-5-mini`).
> Models update quickly — you are welcome (and encouraged) to swap in any newer OpenAI / Anthropic / Google model you have access to.
> The default model is set in one place: `DEFAULT_CHAT_MODEL` inside `utils.py`. Change it there and every cell follows.


In [ ]:
# === Shared lab setup: utils.py + API key + sticky lab pill ===
# Downloads the shared utilities (pretty_print, model constants, key loader,
# lab_pill) from the AppliedGenAI repo so every notebook stays small and
# consistent. The API key is read from a Colab secret named OPENAI_API_KEY
# (set it once under Colab → 🔑 → "Notebook access" — same name in every lab).
import os
if not os.path.exists("utils.py"):
    !wget -q https://raw.githubusercontent.com/mdehghani86/AppliedGenAI/main/utils.py -O utils.py

from utils import (
    pretty_print,
    DEFAULT_CHAT_MODEL,   # e.g. "gpt-5"  — main reasoning model
    DEFAULT_MINI_MODEL,   # e.g. "gpt-5-mini"  — cheaper / faster default
    DEFAULT_EMBED_MODEL,  # e.g. "text-embedding-3-small"
    get_openai_key,
    lab_pill,
)

lab_pill('Prompting with the OpenAI API')        # sticky banner so you always see which lab you're in
get_openai_key(verify=True)    # loads the key + pings OpenAI to confirm it works


KeyboardInterrupt: Interrupted by user

# Applied Generative AI
Instructor: Prof. Dehghani
Welcome to the Applied Generative AI course. In this course, we will explore the foundations and applications of Generative AI using tools like OpenAI's API. By the end of this session, you will:

🟢 Understand how to set up and connect to OpenAI's API.
🌟 Learn about the roles (System, Assistant, User) in prompt design.
✨ Generate text, images, and vector embeddings programmatically.
🔧 Explore fine-tuning to customize AI models for specific tasks.
Let’s get started with setting up the OpenAI API!

In [ ]:
# Install the OpenAI Python SDK
# This library allows us to interact with OpenAI's API for text, images, and embeddings.
!pip install openai==0.28

In [ ]:
from google.colab import userdata
import os

# Retrieve the key from Colab secrets
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Confirm the key was loaded (for debugging only; don’t print your real key in shared notebooks!)
print("Key loaded:", "Yes" if os.environ.get("OPENAI_API_KEY") else "No")

🎯 Prompt Playground: Understanding Roles
In OpenAI's API, you interact with the model using roles, which define the flow of the conversation:

➡️ System: Sets the behavior and tone of the assistant (e.g., "You are a cheerful assistant.").
➡️ User: Represents the input or question from the user (e.g., "What is AI?").
➡️ Assistant: Automatically generated responses based on the system and user inputs.
💡 Why Roles Matter: Roles help control the assistant's personality and the quality of responses. For example:

A system message like "You are a strict teacher" makes the assistant respond more formally.
A system message like "You are a friendly chatbot" leads to casual responses.
Let’s see how these roles work in the next cell!

## Example 1: Without Assistant Role

In [ ]:
from openai import OpenAI
client = OpenAI()

import openai

response = client.chat.completions.create(
    model=DEFAULT_MINI_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France?"}
    ]
)

pretty_print(response.choices[0].message.content, title="🤖 Model Response")

# Example 2: With Assistant Role

Here I would ask the assistant if there was a more consides or different method to follow for the math problem. This seems like a great way to expand

In [ ]:
from openai import OpenAI
client = OpenAI()

# Demonstrating roles in OpenAI's API with the assistant role

# Define the conversation including a predefined assistant response
messages = [
    {"role": "system", "content": "You are a math tutor who explains problems step by step."},  # System role sets the behavior
    {"role": "user", "content": "Solve for x: 2x + 5 = 15"},  # User question
    {"role": "assistant", "content": "To solve for x: \n1. Subtract 5 from both sides: 2x = 10\n2. Divide both sides by 2: x = 5"}  # Predefined assistant response
]

# Send the conversation to the API
response = client.chat.completions.create(
    model=DEFAULT_CHAT_MODEL,  # Use the chosen model
    messages=messages  # Pass the conversation
)

# Print the assistant's response
pretty_print(response.choices[0].message.content, title="Assistant's Response:")

🧮 Hands-



In [ ]:
from openai import OpenAI
client = OpenAI()

messages = [
    {"role": "system", "content": "You are a math tutor who explains concepts clearly and step by step."},
    {"role": "user", "content": "How do you calculate the arithmetic mean of a set of numbers?"},

    # I dont fully understand why we have this in here if it will not read this.
    {"role": "assistant", "content": "To calculate the arithmetic mean:\n"
                                     "1. Add all the numbers in the set.\n"
                                     "2. Divide the sum by the number of numbers.\n\n"
                                     "For example, for 10, 20, and 30:\n"
                                     "Mean = (10 + 20 + 30) / 3 = 60 / 3 = 20."}
]

response = client.chat.completions.create(
    model=DEFAULT_CHAT_MODEL,
    messages=messages
)
pretty_print(response.choices[0].message.content, title="Assistant's Response:")


Exercise 1: Simple Q&A with System + User Roles
👉 Task:
Write a prompt that asks the assistant to behave like a science teacher, then ask it a science-related question.


I wonder if you are able to write a code so the computer can ask it differnet types of science teachers to see the different types of responses you are going to get. There seems like a lot of open room with this style of learning.

In [ ]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

messages = [
    {"role": "system", "content": "You are a science teacher."},
    {"role": "user", "content": "What is the difference between single and multi cell organisms?"}
]

response = client.chat.completions.create(
    model=DEFAULT_CHAT_MODEL,
    messages=messages
)

pretty_print(response.choices[0].message.content, title="Assistant's Response:")

✍️ Exercise 2: Give a prompt of your choice.
👉 Task:
Now write your own custom prompt. Use any role, any question — be creative!

As I am training for one right now this seems to be the perfect way to have it pretend it is different conditioning coaches to get it to make the optimal workout plan.

In [ ]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

# Replace with your own prompt idea!
messages = [
    {"role": "system", "content": "You are a swim instructor working at the YMCA."},
    {"role": "user", "content": "What is a good swim program I should follow if I am training for an Olympic triathlon on August 23?"}
]

response = client.chat.completions.create(
    model=DEFAULT_CHAT_MODEL,
    messages=messages
)

pretty_print(response.choices[0].message.content, title="Assistant's Response:")
